# P38 — Bayes variacional con autocodificación

## 1. Título y paper

**Paper:** *Auto-Encoding Variational Bayes*  
**Autoría:** Diederik P. Kingma, Max Welling  
**Año y venue:** 2013 · arXiv:1312.6114 · ICLR 2014  
**Nivel:** L3 · **Motor:** `vae`  
**Ficha completa:** [`P38_vae`](../../papers/foundational/P38_vae/README.md)

**Hito:** Hace entrenable un modelo generativo latente: el truco de reparametrización deja pasar el gradiente a través del muestreo.

- [arXiv:1312.6114](https://arxiv.org/abs/1312.6114)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Un modelo generativo con variables latentes exige muestrear, y muestrear es un nodo estocástico que bloquea el gradiente: no se podía entrenar por retropropagación.
2. Ejecutar una implementación mínima de la propuesta: Escribir la muestra como z = μ + σ·ε con ε de una normal fija: el azar queda fuera del camino del gradiente, y se optimiza una cota inferior de la verosimilitud (ELBO).
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P02


## 4. Intuición

Quieres derivar respecto a la media de una distribución de la que estás muestreando. Pero muestrear es un dado: no tiene derivada. El truco es sacar el dado fuera — tirarlo aparte y meter su resultado ya fijado en la fórmula.


## 5. Concepto mínimo

```text
Sin reparametrizar:  z ~ N(μ, σ²)          ← nodo estocástico, gradiente bloqueado
Reparametrizado   :  z = μ + σ·ε,  ε ~ N(0,1)  ← el azar está FUERA del camino

ELBO = E_q[log p(x|z)] − KL(q(z|x) ‖ p(z))
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('vae', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿La distribución de z cambia al reparametrizar?
2. ¿Cuánto vale ∂z/∂μ?
3. ¿Por qué eso hace entrenable el modelo?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('vae', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('vae', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

La distribución es la misma —media y varianza coinciden— pero ahora `∂z/∂μ = 1` existe y se puede estimar, porque `ε` no depende de los parámetros. Ese es todo el truco, y es lo que hizo entrenable una familia entera de modelos.


## 10. Comentario pedagógico

El truco de reparametrización trasciende al VAE: aparece en política estocástica, en atención con puertas, en cuantización diferenciable. Cuando algo «no es derivable», la pregunta útil es si se puede reescribir moviendo el azar fuera.


## 11. Error o anti-patrón deliberado

Anti-patrón: creer que el término KL es un detalle de regularización opcional.


In [ ]:
print('Sin el termino KL, el codificador puede mapear cada x a una gaussiana')
print('estrechisima y separada de las demas: el espacio latente deja de ser')
print('continuo y muestrear de la prior ya no genera nada coherente.')

## 12. Corrección

El ELBO tiene dos términos y ambos hacen falta:


In [ ]:
elbo = {'reconstruccion': 'que el decodificador recupere x desde z',
        'KL': 'que q(z|x) no se aleje de la prior, para que el espacio sea muestreable',
        'tension': 'demasiada KL → muestras borrosas; poca → espacio latente roto'}
show(elbo)

## 13. Desafío guiado

Comprueba que la varianza empírica de las muestras coincide con σ² y que el gradiente respecto a σ es insesgado.


In [ ]:
r = run_paper_lab('vae', seed=3)['result']
show(r)

## 14. Desafío autónomo

Implementa un VAE sobre un conjunto de imágenes pequeño y visualiza el espacio latente en 2D. Interpola entre dos puntos y comprueba si las muestras intermedias son coherentes.


## 15. Evidencia de aprendizaje

Guarda la comprobación de media y varianza, el valor del gradiente respecto a μ y tu explicación de los dos términos del ELBO.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P38_vae/README.md) · evaluación formal: [`assessments/papers/P38_vae.md`](../../assessments/papers/P38_vae.md)


## 16. Cierre

Ya se puede entrenar un modelo generativo latente, pero sus muestras son borrosas. La respuesta de 2014 fue radicalmente distinta: convertirlo en un juego.


## 17. Conexión con el siguiente hito

- P39
- P17

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
